# Library Import

In [1]:
from stable_baselines3 import TD3
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_util import make_vec_env
import gymnasium as gym
import os, re

# TD3 training with SB3

In [ ]:
env = gym.make("HalfCheetah-v5", render_mode='rgb_array')
env = make_vec_env("HalfCheetah-v5", n_envs=8, vec_env_cls=DummyVecEnv)
model = TD3("MlpPolicy", env, 
            learning_rate=5e-4,        # lr for all networds - Q-values, Actor, Value function
            buffer_size=500_000,      # replay buffer size
            learning_starts=10_000,        # # of data collection step before training
            batch_size=256,
            tau=5e-3,                  # polyak update coefficient
            gamma=0.99,
            train_freq=1,
            gradient_steps=2, 
            action_noise=None, 
            n_steps=1,                  # n-step TD learning
            policy_delay=2,             # the policy and target networks are updated every policy_delay steps
            target_policy_noise=0.1,   # stdev of noise added to target policy
            target_noise_clip=0.2,      # limit of asbsolute value of noise
            verbose=2)
model.learn(total_timesteps=1_000_000)

Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -257     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 19558    |
|    time_elapsed    | 0        |
|    total_timesteps | 8000     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -257     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 19529    |
|    time_elapsed    | 0        |
|    total_timesteps | 8000     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -337     |
| time/              |          |
|    episodes        | 12       |
|    fps             | 4075     |
|    time_elapsed    | 3        |
|    total_timesteps | 16000  

# Save the Trained Model

In [3]:
BASE_DIR = os.getcwd()
RESULT_FOLDER = 'halfcheetah_TD3_SB_results'
RESULT_DIR = os.path.join(BASE_DIR, RESULT_FOLDER)
existing_runs = [d for d in os.listdir(RESULT_DIR) if os.path.exists(os.path.join(RESULT_DIR,d))]
run_numbers = [int(re.search(r'run_(\d{5})',d).group(1)) for d in existing_runs if re.match(r'run_\d{5}',d)]
# model.save('reacher')

trial_number = max(run_numbers, default=-1) + 1
model.save(f'{RESULT_FOLDER}/run_{trial_number:05d}')

# Simulate the Loaded Model


In [6]:
model_load = TD3.load('halfcheetah_TD3_SB_results/run_00000')

width = 1920
height = 1080
default_camera_config = {"azimuth" : 90.0, "elevation" : 0.0, "distance" : 20, "lookat" : [10.0, 0.0, 1.0]}
camera_id = 2

vec_env = gym.make("HalfCheetah-v5", render_mode='human', 
                    width=width,height=height,
                    default_camera_config=default_camera_config,
                    camera_id=camera_id,
                    frame_skip=1,
                    # camera_name="camera",
                    # max_episode_steps=100
                    )

for eps in range(1):
    obs, _ = vec_env.reset()
    dones = False

    for step in range(1000):
        action, _ = model_load.predict(obs, deterministic=True)
        nobs, rewards, dones, info, _ = vec_env.step(action)
        obs = nobs if not dones else vec_env.reset()
        # vec_env.render("human")

vec_env.close()